#### # 1. Define Gold Product Dimension Logical Query

In [0]:
gold_query = """
SELECT
    ROW_NUMBER() OVER(ORDER BY pn.start_date, pn.product_number) AS product_key,
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance,
    pn.product_cost AS cost, -- Fixed to product_cost
    pn.product_line,
    pn.start_date
FROM workspace.silver.crm_product AS pn
LEFT JOIN workspace.silver.erp_product_category AS pc
    ON pn.category_id = pc.id
WHERE pn.end_date IS NULL
"""

# Compile the query string into the Spark engine
df = spark.sql(gold_query)

#### # 3. Commit Schema to Target Storage & Verify Results

In [0]:
# Save directly as a high-performance Gold Delta Table with atomic overwrite
df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_products")

# Display a clean data sample preview to verify integrity and join logic
df.display()